# Day 048 Solution — Feature Engineering & Intro ML

Demonstrates: one-hot encoding, StandardScaler (fit on train only), LinearRegression, R² + RMSE, feature coefficient analysis, and a saved actual-vs-predicted scatter plot.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os


In [ ]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')


def make_regression_data(n: int = 200, seed: int = 42) -> pd.DataFrame:
    """Synthetic housing dataset with one categorical column (neighborhood)."""
    rng = np.random.default_rng(seed)
    area         = rng.uniform(500, 3000, n).round(0)
    bedrooms     = rng.integers(1, 6, n)
    age          = rng.uniform(0, 50, n).round(1)
    neighborhood = rng.choice(['downtown', 'suburb', 'rural'], n)
    price = (
        area * 150
        + bedrooms * 10_000
        - age * 1_000
        + np.where(neighborhood == 'downtown', 50_000, 0)
        + np.where(neighborhood == 'suburb',   20_000, 0)
        + rng.standard_normal(n) * 10_000
    ).round(-2)
    return pd.DataFrame({
        'area':         area.astype(int),
        'bedrooms':     bedrooms,
        'age':          age,
        'neighborhood': neighborhood,
        'price':        price.astype(int),
    })


def prepare_features(df: pd.DataFrame, target_col: str,
                     numeric_only: bool = True):
    """Return (X, y) separating features from target."""
    X = df.drop(columns=[target_col])
    if numeric_only:
        X = X.select_dtypes(include='number')
    y = df[target_col]
    return X, y


def split_data(X: pd.DataFrame, y: pd.Series,
               test_size: float = 0.2,
               random_state: int = 42) -> dict:
    """Wrap train_test_split, return a result dict."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    return {
        'X_train':    X_train,
        'X_test':     X_test,
        'y_train':    y_train,
        'y_test':     y_test,
        'n_train':    len(X_train),
        'n_test':     len(X_test),
        'n_features': X_train.shape[1],
    }


def encode_categoricals(df: pd.DataFrame,
                         cat_cols: list | None = None,
                         drop_first: bool = False) -> pd.DataFrame:
    """One-hot encode categorical columns with pd.get_dummies."""
    if cat_cols is None:
        cat_cols = df.select_dtypes(include='object').columns.tolist()
    if not cat_cols:
        return df.copy()
    encoded = pd.get_dummies(df, columns=cat_cols, drop_first=drop_first)
    # pandas 2.x returns bool dtype for dummy columns; convert to int
    bool_cols = encoded.select_dtypes(include='bool').columns.tolist()
    for c in bool_cols:
        encoded[c] = encoded[c].astype(int)
    return encoded


from sklearn.preprocessing import StandardScaler


def fit_scaler(X_train: pd.DataFrame) -> StandardScaler:
    """Fit a StandardScaler on training data only."""
    scaler = StandardScaler()
    scaler.fit(X_train)
    return scaler


def scale_features(scaler: StandardScaler,
                   X: pd.DataFrame) -> pd.DataFrame:
    """Transform X using a fitted scaler; return DataFrame with same columns."""
    scaled = scaler.transform(X)
    return pd.DataFrame(scaled, columns=X.columns, index=X.index)


from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score


def train_model(X_train: pd.DataFrame,
                y_train: pd.Series) -> LinearRegression:
    """Fit LinearRegression on training data."""
    model = LinearRegression()
    model.fit(X_train, y_train)
    return model


def evaluate_model(model: LinearRegression,
                   X_test: pd.DataFrame,
                   y_test: pd.Series) -> dict:
    """Return R², RMSE, n_test, and predictions array."""
    y_pred = model.predict(X_test)
    r2     = r2_score(y_test, y_pred)
    rmse   = float(np.sqrt(mean_squared_error(y_test, y_pred)))
    return {
        'r2':          round(float(r2), 4),
        'rmse':        round(rmse, 2),
        'n_test':      len(y_test),
        'predictions': y_pred,
    }


class FeatureEngineer:
    """
    End-to-end preprocessing pipeline: encode → scale → split.

    Usage:
        fe    = FeatureEngineer(target_col='price')
        split = fe.fit_transform(df)
        X_new = fe.transform(new_df)
    """

    def __init__(self, target_col: str,
                 cat_cols: list | None = None,
                 scale: bool = True):
        self.target_col   = target_col
        self.cat_cols     = cat_cols
        self.scale        = scale
        self._scaler      = None
        self._feature_cols = None

    def fit_transform(self, df: pd.DataFrame,
                      test_size: float = 0.2,
                      random_state: int = 42) -> dict:
        """Encode, scale (fit on train), split. Return split dict."""
        encoded             = encode_categoricals(df, cat_cols=self.cat_cols)
        X, y                = prepare_features(encoded, self.target_col,
                                               numeric_only=False)
        self._feature_cols  = X.columns.tolist()
        if self.scale:
            self._scaler = fit_scaler(X)
            X            = scale_features(self._scaler, X)
        return split_data(X, y, test_size=test_size,
                          random_state=random_state)

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """Apply fitted encoding + scaling to new data."""
        encoded = encode_categoricals(df, cat_cols=self.cat_cols)
        X       = encoded.reindex(columns=self._feature_cols, fill_value=0)
        if self.scale and self._scaler is not None:
            X = scale_features(self._scaler, X)
        return X

## Step 1 — Dataset

In [ ]:
df = make_regression_data(200)
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
print(df.head())
print('\nprice stats:')
print(df['price'].describe().round(0))

## Step 2 — Encode + Scale + Split

In [ ]:
fe    = FeatureEngineer(target_col='price')
split = fe.fit_transform(df)

print(f'Train: {split["n_train"]} rows  Test: {split["n_test"]} rows')
print(f'Features ({split["n_features"]}): {split["X_train"].columns.tolist()}')
print(f'\nX_train stats (scaled):')
print(split['X_train'].describe().round(4))

assert split['n_train'] + split['n_test'] == len(df)
assert 'price' not in split['X_train'].columns

## Step 3 — Train

In [ ]:
model = train_model(split['X_train'], split['y_train'])
print('Model trained.')
print('Intercept:', round(model.intercept_, 2))

assert hasattr(model, 'coef_')
assert len(model.coef_) == split['n_features']

## Step 4 — Evaluate

In [ ]:
result = evaluate_model(model, split['X_test'], split['y_test'])

r2   = result['r2']
rmse = result['rmse']
print(f'R\u00b2  = {r2:.4f}')
print(f'RMSE = ${rmse:,.2f}')
print(f'Test samples: {result["n_test"]}')

assert r2 > 0.9, f'expected R\u00b2 > 0.9, got {r2}'

## Step 5 — Feature Importance

In [ ]:
coef_df = pd.DataFrame({
    'feature':     split['X_train'].columns.tolist(),
    'coefficient': model.coef_,
}).sort_values('coefficient', key=abs, ascending=False)

print('Feature Coefficients (scaled units):')
print(coef_df.to_string(index=False))
print('\nLargest positive driver:', coef_df.iloc[0]['feature'])

## Step 6 — Save Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel 1: actual vs predicted scatter
lo = min(float(split['y_test'].min()), float(result['predictions'].min()))
hi = max(float(split['y_test'].max()), float(result['predictions'].max()))
axes[0].scatter(split['y_test'], result['predictions'],
                alpha=0.65, edgecolors='white', linewidths=0.3)
axes[0].plot([lo, hi], [lo, hi], 'r--', linewidth=1.5, label='perfect fit')
axes[0].set_xlabel('Actual Price ($)')
axes[0].set_ylabel('Predicted Price ($)')
axes[0].set_title(f'Actual vs Predicted  R\u00b2={r2:.3f}  RMSE=${rmse:,.0f}')
axes[0].legend()

# Panel 2: feature coefficients bar chart
colors = ['steelblue' if c > 0 else 'tomato' for c in coef_df['coefficient']]
axes[1].barh(coef_df['feature'], coef_df['coefficient'], color=colors)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Coefficient (std-unit impact on price)')
axes[1].set_title('Feature Coefficients')

plt.tight_layout()
fig.savefig('predictive_model.png', bbox_inches='tight', dpi=100)
plt.close('all')
print('Chart saved: predictive_model.png')

assert os.path.exists('predictive_model.png')
assert os.path.getsize('predictive_model.png') > 1000

print('\nFeature Engineering & Intro ML complete!')